# Bronze Data Quality

In [1]:
import os
import sys
from pathlib import Path

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "warehouse").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root."
    )


def find_java_home() -> Path:
    # Spark's bundled Hadoop is incompatible with Java 24+, so pin a JDK <= 21.
    candidates = [
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
    ]
    for candidate in candidates:
        if (candidate / "bin" / "java").exists():
            return candidate
    raise FileNotFoundError(
        "No JDK 17/21 found. Install one with: brew install openjdk@21"
    )


PROJECT_ROOT = find_project_root()
os.environ["JAVA_HOME"] = str(find_java_home())
# Workers must run the same Python as the driver, not whatever python3 is on PATH.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

CATALOG_DB = PROJECT_ROOT / "data" / "catalog.db"
WAREHOUSE_DIR = PROJECT_ROOT / "data" / "warehouse"

# iceberg-spark-runtime must match PySpark's Spark/Scala version (4.1 / 2.13).
SPARK_PACKAGES = ",".join(
    [
        "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0",
        "org.xerial:sqlite-jdbc:3.49.1.0",
    ]
)


def create_spark_session() -> SparkSession:
    builder = (
        SparkSession.builder
        .appName("nyc-taxi-lakehouse")
        .master("local[*]")
        # Wide aggregations over 11M rows need more than the default 1g heap.
        .config("spark.driver.memory", "4g")
        .config("spark.jars.packages", SPARK_PACKAGES)
        .config(
            "spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        )
        # Same SQLite catalog the pipelines write to (pyiceberg SqlCatalog).
        .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.local.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
        .config("spark.sql.catalog.local.uri", f"jdbc:sqlite:{CATALOG_DB}")
        .config("spark.sql.catalog.local.warehouse", WAREHOUSE_DIR.as_uri())
        .config("spark.sql.session.timeZone", "UTC")
    )
    try:
        return builder.getOrCreate()
    except Exception:
        # If the previous JVM died (e.g. OutOfMemoryError), getOrCreate keeps
        # reusing the dead session. Drop the stale handles and launch fresh.
        SparkSession._instantiatedSession = None
        SparkSession._activeSession = None
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None
        return builder.getOrCreate()


spark = create_spark_session()
spark.sparkContext.setLogLevel("WARN")

BRONZE_TABLE = "local.bronze.yellow_trips"

print("Project root:", PROJECT_ROOT)
print("Java home:", os.environ["JAVA_HOME"])


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/12 03:23:00 WARN Utils: Your hostname, admins-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.125 instead (on interface en0)
26/07/12 03:23:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ittipat/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ittipat/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
org.xerial#sqlite-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b27b90cd-8e95-466a-b738-f3f907370ef5;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
	found org.xerial#sqlite-jdbc;3.49.1.0 in central
:: resolution report :: 

Project root: /Users/ittipat/Desktop/nyc-taxi-lakehouse-demo
Java home: /opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home


In [2]:
bronze_df = spark.table(BRONZE_TABLE)

pickup_column = "tpep_pickup_datetime"
dropoff_column = "tpep_dropoff_datetime"
row_count = bronze_df.count()

print(f"Bronze rows: {row_count:,}")

26/07/12 03:23:04 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


Bronze rows: 11,077,206


In [3]:
bronze_df.agg(
    F.sum(F.col(pickup_column).isNull().cast("long")).alias("missing_pickup"),
    F.sum(F.col(dropoff_column).isNull().cast("long")).alias("missing_dropoff"),
    F.sum(
        (
            F.col(pickup_column).isNotNull()
            & F.col(dropoff_column).isNotNull()
            & (F.col(dropoff_column) <= F.col(pickup_column))
        ).cast("long")
    ).alias("dropoff_not_after_pickup"),
    F.sum(
        (
            F.col(pickup_column).isNotNull()
            & F.col(dropoff_column).isNotNull()
            & (
                F.col(dropoff_column).cast("timestamp").cast("long")
                - F.col(pickup_column).cast("timestamp").cast("long")
                > 24 * 60 * 60
            )
        ).cast("long")
    ).alias("duration_over_24_hours"),
).show(truncate=False)

+--------------+---------------+------------------------+----------------------+
|missing_pickup|missing_dropoff|dropoff_not_after_pickup|duration_over_24_hours|
+--------------+---------------+------------------------+----------------------+
|0             |0              |134801                  |100                   |
+--------------+---------------+------------------------+----------------------+



In [4]:
misdated_df_by_global_date = bronze_df.filter(
    F.col(pickup_column).isNotNull()
    & (
        (F.to_date(F.col(pickup_column)) < F.lit("2026-01-01").cast("date"))
        | (F.to_date(F.col(pickup_column)) >= F.lit("2026-04-01").cast("date"))
    )
)

print(f"Rows outside source month: {misdated_df_by_global_date.count():,}")
misdated_df_by_global_date.select(
    "_bronze_source_file",
    pickup_column,
    dropoff_column,
).show(20, truncate=False)

Rows outside source month: 10
+-------------------------------+--------------------+---------------------+
|_bronze_source_file            |tpep_pickup_datetime|tpep_dropoff_datetime|
+-------------------------------+--------------------+---------------------+
|yellow_tripdata_2026-03.parquet|2009-01-01 00:02:17 |2009-01-01 00:07:36  |
|yellow_tripdata_2026-03.parquet|2008-12-31 23:03:20 |2009-01-01 14:24:52  |
|yellow_tripdata_2026-03.parquet|2026-04-01 00:00:16 |2026-04-01 00:15:31  |
|yellow_tripdata_2026-03.parquet|2026-04-01 00:06:25 |2026-04-01 00:14:42  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:57:29 |2025-12-31 23:57:32  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:57:29 |2025-12-31 23:57:32  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:58:58 |2026-01-01 01:04:39  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:57:54 |2026-01-01 00:09:36  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:59:06 |2026-01-01 00:16:43  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23

In [5]:
source_month_df = (
    bronze_df
    .withColumn(
        "_source_month_text",
        F.regexp_extract(
            F.col("_bronze_source_file"),
            r"yellow_tripdata_(\d{4}-\d{2})\.parquet$",
            1,
        ),
    )
    .withColumn(
        "_source_month_start",
        F.to_date(F.concat(F.col("_source_month_text"), F.lit("-01"))),
    )
    .withColumn(
        "_source_next_month",
        F.add_months(F.col("_source_month_start"), 1),
    )
)

misdated_df = source_month_df.filter(
    F.col(pickup_column).isNotNull()
    & F.col("_source_month_start").isNotNull()
    & (
        (F.to_date(F.col(pickup_column)) < F.col("_source_month_start"))
        | (F.to_date(F.col(pickup_column)) >= F.col("_source_next_month"))
    )
)

print(f"Rows outside source month: {misdated_df.count():,}")
misdated_df.select(
    "_bronze_source_file",
    pickup_column,
    dropoff_column,
).show(20, truncate=False)

Rows outside source month: 42


+-------------------------------+--------------------+---------------------+
|_bronze_source_file            |tpep_pickup_datetime|tpep_dropoff_datetime|
+-------------------------------+--------------------+---------------------+
|yellow_tripdata_2026-01.parquet|2025-12-31 23:57:29 |2025-12-31 23:57:32  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:57:29 |2025-12-31 23:57:32  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:58:58 |2026-01-01 01:04:39  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:57:54 |2026-01-01 00:09:36  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:59:06 |2026-01-01 00:16:43  |
|yellow_tripdata_2026-01.parquet|2025-12-31 23:58:45 |2026-01-01 00:07:12  |
|yellow_tripdata_2026-03.parquet|2026-04-01 00:00:16 |2026-04-01 00:15:31  |
|yellow_tripdata_2026-03.parquet|2026-04-01 00:06:25 |2026-04-01 00:14:42  |
|yellow_tripdata_2026-03.parquet|2026-02-28 23:58:32 |2026-03-01 00:07:53  |
|yellow_tripdata_2026-03.parquet|2026-02-28 23:58:02 |2026-02-28 23:58:10  |

In [6]:
bronze_df.agg(
    F.sum(
        (F.col("trip_distance").isNull() | (F.col("trip_distance") <= 0)).cast("long")
    ).alias("invalid_trip_distance"),
    F.sum(
        (F.col("fare_amount").isNull() | (F.col("fare_amount") < 0)).cast("long")
    ).alias("invalid_fare_amount"),
    F.sum(
        (F.col("total_amount").isNull() | (F.col("total_amount") < 0)).cast("long")
    ).alias("invalid_total_amount"),
    F.sum(
        (
            F.col("passenger_count").isNotNull()
            & (F.col("passenger_count") <= 0)
        ).cast("long")
    ).alias("invalid_passenger_count"),
).show(truncate=False)

+---------------------+-------------------+--------------------+-----------------------+
|invalid_trip_distance|invalid_fare_amount|invalid_total_amount|invalid_passenger_count|
+---------------------+-------------------+--------------------+-----------------------+
|370198               |86908              |88580               |40638                  |
+---------------------+-------------------+--------------------+-----------------------+



In [11]:
duplicate_key_columns = [
    column
    for column in bronze_df.columns
    if column != "_bronze_ingested_at" and column != "_bronze_source_file"
]

duplicate_groups = (
    bronze_df
    .groupBy(*duplicate_key_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_group_count = duplicate_groups.count()

duplicate_row_count = (
    duplicate_groups
    .select(
        F.sum(F.col("count") - 1).alias("duplicate_rows")
    )
    .first()["duplicate_rows"]
    or 0
)

print(f"Duplicate groups: {duplicate_group_count:,}")
print(f"Duplicate rows: {duplicate_row_count:,}")

Duplicate groups: 0
Duplicate rows: 0


In [9]:
(
    bronze_df
    .groupBy("passenger_count")
    .count()
    .orderBy(F.asc_nulls_first("passenger_count"))
    .show(50, truncate=False)
)

+---------------+-------+
|passenger_count|count  |
+---------------+-------+
|NULL           |3057123|
|0              |40638  |
|1              |6630610|
|2              |968664 |
|3              |206283 |
|4              |134234 |
|5              |25366  |
|6              |14274  |
|7              |2      |
|8              |9      |
|9              |3      |
+---------------+-------+

